# 🤖 Train your first classifier

Machine learning in one sentence: **show the computer examples, let it find the pattern, then test it on examples it has never seen.**

Our task: predict whether a learner will **pass a digital-skills certification**, using the *Student Digital Skills* dataset (synthetic, education-sector themed — 900 learners). This is a **classification** problem: the answer is pass (1) or fail (0).

In [ ]:
import pandas as pd

df = pd.read_csv("data/student_digital_skills.csv")
df.head()

## Features and target

- **Features (X):** what we know *before* the exam — age, study hours, skill score…
- **Target (y):** what we want to predict — `passed_certification`

Notice one deliberate choice: we **leave `has_own_laptop` out of the features**. It feels like the responsible thing to do — if the model never sees who can afford a laptop, it cannot discriminate on it, right? Hold that thought; we will test it later.

In [ ]:
features = ["age", "hours_online_learning", "digital_skill_score",
            "prior_ict_courses"]          # note: has_own_laptop deliberately excluded
X = df[features]
y = df["passed_certification"]

print("Pass rate in the data:", round(y.mean(), 2))

## Split: train vs. test

**The golden rule of ML:** never evaluate a model on the data it learned from — that's like grading students on the exact questions they rehearsed. We hold back 25% as a surprise exam. (The first scikit-learn import takes a few seconds to load.)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(f"Training on {len(X_train)} learners, testing on {len(X_test)}")

## Train a decision tree

Two lines. Really.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)

accuracy = tree.score(X_test, y_test)
print(f"Accuracy on unseen learners: {accuracy:.0%}")

The model now predicts pass/fail for learners it has **never seen**.

Before celebrating, ask the question every honest analyst asks: **what would a lazy baseline score?** If you simply predicted *"everyone passes"*, you would be right about 61% of the time. Our model manages ~66% — a real improvement, but a modest one. Hold on to that mild disappointment: by the end of this notebook you will know exactly where it comes from.

Where does the model go wrong? A **confusion matrix** shows the four possible outcomes:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

ConfusionMatrixDisplay.from_estimator(
    tree, X_test, y_test, display_labels=["failed", "passed"], cmap="Blues"
)
plt.title("Where the model is right and wrong")
plt.show()

## What drives the prediction?

Decision trees can tell us which features mattered most:

In [ ]:
importances = pd.Series(tree.feature_importances_, index=features).sort_values()
importances.plot(kind="barh", color="#1d3fbf")
plt.title("Feature importance — what the model relies on")
plt.show()

## ⚖️ Ethics lab — measure what you built

Imagine a university used this model to decide **who gets a scholarship place**.

We were careful: the model never saw `has_own_laptop`, so it *cannot* be biased by who can afford a computer. This is such a common instinct that it has a name — **fairness through unawareness**.

Most courses would ask you to *reflect* on whether that is enough. Here, we **measure it** — fairness is an empirical question, and you have the tools.

**Beat 1 — Build it** ✅ (you already did). Now:

**Beat 2 — Measure it.** We audit *by* laptop ownership even though the model never used it — which is exactly how real fairness audits work.

In [ ]:
%run datalab_ethics.py

# the audit attribute — present in the data, never given to the model
laptop = df.loc[X_test.index, "has_own_laptop"].map({1: "has laptop", 0: "no laptop"})
group_performance(tree, X_test, y_test, by=laptop)

Read that table slowly. The model is right about **72%** of learners who own a laptop — but only **54%** of those who don't, barely better than guessing. And its *false negatives* (learners who **would have passed** but were predicted to fail) are **twice as common** in the no-laptop group: 38% against 19%.

If this model allocated scholarship places, it would wrongly reject the most disadvantaged applicants twice as often — while never once looking at who owns a laptop.

Now the screen the industry reaches for by default — the **four-fifths rule**: do both groups *receive* positive predictions at similar rates?

In [ ]:
selection_rate_ratio(tree.predict(X_test), groups=laptop)

**Beat 3 — Name it.** Here is the twist that makes this lesson worth an hour of your life. This model:

- never saw the sensitive attribute — *fairness through unawareness* ✅
- gives both groups positive predictions at nearly the same rate — **passes the four-fifths screen** (ratio ≈ 0.91) ✅
- and is **wrong about nearly half** the learners without a laptop, rejecting deserving ones twice as often ❌

An analyst running the two standard checks would sign this off. The harm would sail straight through.

**Why does it happen?** Not because the model is prejudiced — it is because the dataset *cannot describe* these learners. What actually determines whether they pass — a shared family computer, library opening hours, time borrowed around a job — was never recorded. For them the model is close to guessing, and that is also where our disappointing 66% headline accuracy came from.

**Two lessons, and they are the heart of this course:**

1. **Deleting the sensitive column does not create fairness.** It removes your ability to *detect* unfairness while leaving the unfairness in place.
2. **One fairness metric is never enough.** Selection-rate parity and error-rate fairness are different properties; a model can satisfy one while badly violating the other. This is the central finding of a decade of fairness research.

### Mind the jurisdiction

The four-fifths rule you just ran is **American**: it comes from the US EEOC's *Uniform Guidelines on Employee Selection Procedures* (1978). It is not European law, and it travelled the world only because every fairness toolkit ships it.

**EU law gives you no threshold at all.** Two things apply instead:

- **Non-discrimination law** (Directives 2000/43/EC, 2000/78/EC) defines *indirect discrimination* qualitatively — does an apparently neutral rule put a protected group at a *particular disadvantage*, without objective justification? A court weighs the evidence; no number decides it.
- **The EU AI Act** classifies systems governing **access to education** as **high-risk** (Annex III), obliging providers to examine data for bias (Art. 10), provide transparency (Art. 13) and ensure human oversight (Art. 14) — yet it prescribes **no fairness metric**.

So Europe hands you *duties*, not a pass mark. Which measure to trust, and what gap is too large, is a judgement you must make and defend — which is precisely why the next beat exists.

In Digital Humanism terms (the *Vienna Manifesto*): the model optimised accuracy but encoded an economic inequality as if it were ability — technology deciding who gets a chance, which is exactly what human-centred design exists to prevent.

**Beat 4 — Judge it.** Your call, in writing. This is the skill employers and regulators actually need:

In [ ]:
my_decision = """
Your answer: would you deploy this model to allocate scholarships? Why / why not?
What single change would most reduce the harm you measured above?
"""

reflect_submit("ml-classifier-ethics", my_decision)

*(Your judgment is now part of your DataLab portfolio. `show_portfolio()` lists what you have saved so far; the Track 3 capstone exports all of it as one file.)*

## Experiment

- Change `max_depth` to 2 or 10 and re-run — does test accuracy improve or get worse?
- **Add `has_own_laptop` back into `features`** and re-run the whole notebook, ethics lab included. Does giving the model the sensitive attribute make the audit better or worse? (There is no comfortable answer — that is the point.)
- Train on *only* the learners who own a laptop. Does the accuracy gap grow?
- Try `from sklearn.linear_model import LogisticRegression` instead of the tree — does a different model rescue the group this data cannot describe?

## ✅ What you learned

Features vs. target · train/test split · fitting a classifier · accuracy against a baseline · confusion matrix · feature importance · **auditing group fairness** · why *fairness through unawareness* fails · why one fairness metric is never enough · and that EU law gives you duties rather than a threshold.

📚 **Go further:**
- [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html) — every model, explained
- [Fairlearn](https://fairlearn.org/) — the open-source toolkit for measuring fairness

EURIDICE's *AI Ethics & Digital Humanism* course goes deep on everything you just touched: bias, fairness metrics and their trade-offs, accountability, and the regulatory landscape.

**Next:** *Overfitting, and choosing a model* — is a better model a fairer one?